# 해협별 운임비 변동 데이터_12주
홍해·호르무즈 사건일 기준 **사건 전 12주(평시)**와 **사건 후 12주**의 KCCI 주간 운임비, 사건후 일수, 전주비를 엑셀로 정리합니다.

- 입력: `financedata/KCCI.xls`
- 출력: `해협별 운임비 변동 데이터_12주.xlsx` (요약 / 항로별_초크포인트 / 사건_반응_타임라인)

In [ ]:
from pathlib import Path
import pandas as pd

DESK = Path.home() / "OneDrive" / "Desktop"
DATA = DESK / "financedata"   # 입력 데이터
KEYS = DESK / "finance"       # API 키, KOSIS·관세청 원자료
SAVE = DESK / "finance"       # 결과 저장 위치 

from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

## 1. 요약·항로별 초크포인트 시트 내용

In [ ]:
SUMMARY = [['홍해·호르무즈 해협 사태 — 운임 반응 정리', None, None], ['작성: 2026-09-28  |  자료: KCCI(한국해양진흥공사) 주간 컨테이너 운임지수, IMF PortWatch', None, None], [None, None, None], ['핵심 결론', None, None], ['•  호르무즈(2026.3.1 봉쇄)는 사건 후 8일 만에 운임 급등 시작 — 실제 물리적 봉쇄라 선박이 즉시 우회', None, None], ['•  홍해(2023.11.19 나포)는 사건 후 29일 만에 급등 시작 — 해협은 열려있었으나 선사들이 위험 판단에 시간 소요', None, None], ["•  두 사건 모두 '해당 항로만' 급등하고 무관한 항로는 안정적 — 지정학 충격이 항로별로 특이하게 나타남", None, None], ['•  상세 근거는 시트 2(항로·초크포인트 설명), 시트 3(사건별 반응 타임라인) 참조', None, None], [None, None, None], ['구분', '호르무즈 해협(2026)', '홍해·바브엘만데브(2023~24)'], ['사건 발생일', '2026-03-01 (이란, 해협 봉쇄)', '2023-11-19 (후티, 갤럭시리더호 나포)'], ['실제 상태', '물리적 봉쇄 (통항 불가)', '위협 (통항 가능하나 위험, 자발적 우회)'], ['관련 항로(KCCI)', 'KMEI (중동향)', 'KNEI (유럽향)'], ['PortWatch 통항량 변화', '-91~95% (사실상 전면 중단)', '-78% (감소하나 0은 아님, 중·러 선박 계속 통항)'], ['운임 급등 시작', '사건 후 +8일 (전주비 +75.7%)', '사건 후 +29일 (전주비 +15.2%)'], ['정점 도달', '약 5주 (+36일)', '약 9주 (+64일)'], ['최대 주간 상승폭', '+75.7%', '+49.6%'], ['반응 속도가 다른 이유', '배가 못 지나가 즉시 우회할 수밖에 없음', '선사가 위험을 판단하는 데 시간 소요(JWC 고위험지역 지정, 전쟁위험보험료 급등, IBF 선원 항해거부권 등)']]

CHOKE = [['KCCI 13개 항로 — 경유 초크포인트(해협·운하) 및 화물 설명', None, None, None, None, None, None, None], ['자료: 한국해양진흥공사(KOBC) KCCI 컬럼설명 + 무역협회 TradeBrief(2024.2) 품목별 해상비중', None, None, None, None, None, None, None], [None, None, None, None, None, None, None, None], ['코드', '분류', '항로명(목적지)', '경유 초크포인트', '초크포인트 유형', '이번 사건 직접 노출', '주요 수출 화물', '주요 수입 화물'], ['KUWI (미서안향)', '메인레인', '미서안(미국 서해안)', '없음(태평양 직항)', '-', '해당없음', '자동차·자동차부품·가전제품·반도체 장비', '농산물(밀·콩 등 일부)·항공기부품·의료기기'], ['KUEI (미동안향)', '메인레인', '미동안(미국 동해안)', '파나마 운하(부분)', '운하', '간접(파나마 가뭄 등 별도 이슈)', '자동차·자동차부품·기계류', '농산물·화학원료'], ['KNEI (유럽향)', '메인레인', '유럽', '수에즈 운하 · 바브엘만데브 해협', '운하+해협', '직접(홍해 사태)', '자동차·2차전지(배터리)·선박부품·화학제품', '의약품·명품 소비재·자동차부품·화학원료'], ['KMDI (지중해향)', '메인레인', '지중해', '수에즈 운하', '운하', '직접(홍해 사태, KNEI와 유사 경로)', '자동차·기계류·전자제품', '의류·농산물(올리브유 등)'], ['KMEI (중동향)', '논메인레인', '중동', '호르무즈 해협', '해협', '직접(호르무즈 봉쇄)', '자동차·건설기계·가전제품·전선케이블', '석유화학 중간재·대추야자 등'], ['KAUI (호주향)', '논메인레인', '호주', '없음(대양 직항)', '-', '해당없음', '자동차·기계류·전자제품', '육류·와인·양모'], ['KLEI (중남미동안향)', '논메인레인', '중남미 동안', '파나마 운하', '운하', '간접', '자동차·가전제품', '커피·과일·광물 일부'], ['KLWI (중남미서안향)', '논메인레인', '중남미 서안', '없음(태평양 직항)', '-', '해당없음', '자동차·가전제품', '구리·과일'], ['KSAI (남아공향)', '논메인레인', '남아공', '없음(홍해 우회선 몰리는 항로)', '-', '간접(희망봉 우회선 집중)', '자동차·기계류', '와인·과일·광물 일부'], ['KWAI (서아프리카향)', '논메인레인', '서아프리카', '없음(희망봉 우회항로와 인접)', '-', '간접(희망봉 우회선 집중)', '자동차·소비재·기계류', '카카오'], ['KCI (중국향)', '인트라아시아', '중국', '없음(대만해협 인근 잠재적)', '-', '해당없음', '반도체 소재·부품·장비·석유화학 중간재', '전자제품·의류·생활용품'], ['KJI (일본향)', '인트라아시아', '일본', '없음', '-', '해당없음', '반도체 소재·부품·장비·자동차부품', '정밀기계·반도체 장비·소재'], ['KSEI (동남아향)', '인트라아시아', '동남아', '말라카 해협', '해협', '잠재적(대만해협 유사시 영향권, 이번 두 사건과는 무관)', '반도체·전자부품·자동차부품', '의류·신발·가공식품·원자재 중간재'], [None, None, None, None, None, None, None, None], [None, None, None, None, None, None, None, None], ['5대 해상 초크포인트(해협·운하) 상세', None, None, None, None, None, None, None], ['초크포인트', '구분', '위치', '규모', '한국 의존도', None, None, None], ['호르무즈 해협', '해협', '이란-오만 사이', '최협부 33km', '원유수입의 69.6~70%, 가스수입의 15%', None, None, None], ['말라카 해협', '해협', '말레이시아-인도네시아 사이', '최협부 2.8km', '수출입 물동량의 30%↑, 원유수입의 90%', None, None, None], ['수에즈 운하', '운하', '이집트 관통', '폭 205m', '대유럽 수출입 경로(정확한 비율 미확보)', None, None, None], ['바브엘만데브 해협', '해협', '예멘-지부티/에리트레아 사이', '26~32km', '수에즈 진입 관문(수에즈와 동일 취급)', None, None, None], ['파나마 운하', '운하', '중미 관통', '길이 82km', '이용국 순위 6~8위, 미국 걸프만 발 원유·LNG 수입 의존', None, None, None], [None, None, None, None, None, None, None, None], ['주: 말라카 해협의 원유 의존도(90%)가 호르무즈(70%)보다 높음 — 중동산 원유가 호르무즈 통과 후 반드시 말라카도 통과해야 하기 때문', None, None, None, None, None, None, None]]

SHEET_LAYOUT = {
    "요약": {"rows": SUMMARY, "merge": ["A1:F1", "A2:F2", "A5:F5", "A6:F6", "A7:F7", "A8:F8"],
             "width": {"A": 22.2, "B": 40.4, "C": 46.4}, "head_rows": [10], "sec_rows": [4]},
    "항로별_초크포인트": {"rows": CHOKE, "merge": ["A1:H1", "A2:H2", "A28:E28"],
             "width": {"A": 18, "B": 12.1, "C": 16.1, "D": 24.2, "E": 12.1, "F": 30.3, "G": 30.3, "H": 30.3},
             "head_rows": [4, 21], "sec_rows": [20]},
}

## 2. KCCI 주간 운임비와 사건 정보

In [ ]:
kcci = pd.read_excel(DATA / "KCCI.xls", header=1)
kcci["DATE"] = pd.to_datetime(kcci["DATE"].astype(str), format="%Y%m%d")
kcci = kcci.sort_values("DATE").reset_index(drop=True)

PRE_DAYS = 84   # 사건 전 12주 = 평시 기준 구간
EVENTS = [
    {"title": "① 호르무즈 봉쇄 — KMEI(중동향), 사건일 2026-03-01", "event": "2026-03-01",
     "code": "KMEI", "route": "KMEI (중동향)", "last": "2026-05-26"},
    {"title": "② 홍해 사태 — KNEI(유럽향), 사건일 2023-11-19", "event": "2023-11-19",
     "code": "KNEI", "route": "KNEI (유럽향)", "last": "2024-02-19"},
]
NOTES = {
    "2026-03-09": "★ 전주비 +75.7%, 급등 시작",
    "2026-04-13": "◆ 정점 부근",
    "2026-05-11": "05-04 조사 없음(연휴) → 2주 전 대비",
    "2026-05-26": "화요일 조사(05-25 대체공휴일), 사건후 약 한 분기",
    "2023-12-18": "★ 전주비 +15.2%, 급등 시작",
    "2024-01-29": "◆ 정점 부근",
}

blocks = []
for e in EVENTS:
    ev = pd.Timestamp(e["event"])
    sel = kcci[(kcci["DATE"] >= ev - pd.Timedelta(days=PRE_DAYS)) & (kcci["DATE"] <= pd.Timestamp(e["last"]))]
    rows = [{"조사일": d, "운임비": int(v),
             "비고": "평시 기준 구간" if d < ev else NOTES.get(f"{d:%Y-%m-%d}", "")}
            for d, v in zip(sel["DATE"], sel[e["code"]])]
    blocks.append({**e, "event": ev, "rows": rows})
    print(e["route"], "행 수:", len(rows), "| 평시 운임비:", round(sel.loc[sel["DATE"] < ev, e["code"]].mean()))

## 3. 엑셀 작성 (서식·수식 포함)

In [ ]:
F = "맑은 고딕"
TITLE = Font(name=F, size=14, bold=True)
SUB = Font(name=F, size=9, color="FF595959")
BOLD = Font(name=F, size=10, bold=True)
HEAD = Font(name=F, size=10, bold=True, color="FFFFFFFF")
BODY = Font(name=F, size=10)
BLUE = Font(name=F, size=10, color="FF0000FF")   # 원자료 입력값
HEAD_FILL = PatternFill("solid", fgColor="FF1F4E79")
SEC_FILL = PatternFill("solid", fgColor="FFDDEBF7")
BASE_FILL = PatternFill("solid", fgColor="FFFFF2CC")
THIN = Side(style="thin", color="FFBFBFBF")
BOX = Border(left=THIN, right=THIN, top=THIN, bottom=THIN)

wb = Workbook()
wb.remove(wb.active)

for name, lay in SHEET_LAYOUT.items():
    ws = wb.create_sheet(name)
    ncol = len(lay["rows"][0])
    for r, row in enumerate(lay["rows"], start=1):
        for c, v in enumerate(row, start=1):
            if v is not None:
                cell = ws.cell(r, c, v)
                cell.font = BODY
                cell.alignment = Alignment(vertical="center", wrap_text=True)
    for r in lay["head_rows"]:
        for c in range(1, ncol + 1):
            if ws.cell(r, c).value is not None:
                ws.cell(r, c).font, ws.cell(r, c).fill = HEAD, HEAD_FILL
        rr = r
        while rr <= ws.max_row and any(ws.cell(rr, c).value is not None for c in range(1, ncol + 1)):
            for c in range(1, ncol + 1):
                if ws.cell(r, c).value is not None:
                    ws.cell(rr, c).border = BOX
            rr += 1
    for r in lay["sec_rows"]:
        ws.cell(r, 1).font = BOLD
    for m in lay["merge"]:
        ws.merge_cells(m)
    ws["A1"].font, ws["A2"].font = TITLE, SUB
    for col, w in lay["width"].items():
        ws.column_dimensions[col].width = w
    ws.sheet_view.showGridLines = False

ws = wb.create_sheet("사건_반응_타임라인")
ws["A1"] = "사건별 KCCI 주간 운임 반응 — 실측 데이터"
ws["A1"].font = TITLE
ws["A2"] = ("파란 글씨=KCCI 원자료 입력값, 검은 글씨=수식. 노란 행=평시 기준 구간(사건 전 12주). "
            "평시 운임비=사건 전 12주 조사치 평균. 자료: financedata/KCCI.xls")
ws["A2"].font = SUB
ws.merge_cells("A1:F1")
ws.merge_cells("A2:F2")


def write_block(top, b):
    for c in range(1, 6):
        ws.cell(top, c).fill = SEC_FILL
    ws.cell(top, 1, b["title"]).font = BOLD
    ev_row, base_row, hr = top + 1, top + 2, top + 4
    ws.cell(ev_row, 1, "사건일").font = BOLD
    x = ws.cell(ev_row, 2, b["event"].to_pydatetime())
    x.font, x.number_format = BLUE, "yyyy-mm-dd"
    ws.cell(base_row, 1, "평시 운임비").font = BOLD
    ws.cell(base_row, 3, "← 사건 전 12주 조사치 평균").font = SUB
    for c, h in enumerate(["조사일", b["route"], "사건후 일수", "전주비(%)", "비고"], start=1):
        x = ws.cell(hr, c, h)
        x.font, x.fill, x.border = HEAD, HEAD_FILL, BOX
        x.alignment = Alignment(horizontal="center")
    pre_rows = []
    for i, row in enumerate(b["rows"]):
        r = hr + 1 + i
        is_pre = row["조사일"] < b["event"]
        if is_pre:
            pre_rows.append(r)
        a = ws.cell(r, 1, row["조사일"].to_pydatetime())
        a.font, a.number_format = BLUE, "yyyy-mm-dd"
        v = ws.cell(r, 2, row["운임비"])
        v.font, v.number_format = BLUE, "#,##0"
        cc = ws.cell(r, 3, f"=A{r}-$B${ev_row}")
        cc.font, cc.number_format = BODY, "0"
        d = ws.cell(r, 4, "-" if i == 0 else f"=(B{r}-B{r-1})/B{r-1}")
        d.font, d.number_format = BODY, "+0.0%;-0.0%"
        ws.cell(r, 5, row["비고"]).font = BODY
        for c in range(1, 6):
            ws.cell(r, c).border = BOX
            ws.cell(r, c).alignment = Alignment(vertical="center", horizontal="left" if c == 5 else "center")
            if is_pre:
                ws.cell(r, c).fill = BASE_FILL
    x = ws.cell(base_row, 2, f"=AVERAGE(B{pre_rows[0]}:B{pre_rows[-1]})")
    x.font, x.number_format = BOLD, "#,##0"
    return hr + 1 + len(b["rows"])


nxt = write_block(4, blocks[0])
write_block(nxt + 2, blocks[1])
for col, w in {"A": 14.1, "B": 13, "C": 12.1, "D": 12.1, "E": 52}.items():
    ws.column_dimensions[col].width = w
ws.sheet_view.showGridLines = False

out = SAVE / "해협별 운임비 변동 데이터_12주.xlsx"
wb.save(out)
print("저장:", out)